# Extraction comparison: neuropil and background corrections for one run

The QC notebook's ΔF/F0 subtracts 0.7 of each ROI's neuropil (Suite2p's default), and in the good session about half the ROIs then have F0 ≤ 0 somewhere. That default assumes the neuropil ring carries calcium signal about as bright as the contamination inside the ROI. With **ribosome-tethered GCaMP** the indicator stays in the somata, so the neuropil holds little of it: its signal is mostly out-of-focus somata, autofluorescence and any detector offset, and subtracting 0.7 of it can remove most of a dim cell's own baseline.

This notebook loads one run that has already been registered and extracted with the QC notebook (`F.npy`, `Fneu.npy`, `stat.npy` from the Cellpose masks), and compares, on the same ROIs:

| Correction | Corrected F |
|---|---|
| Neuropil 0.7 | `F − 0.7·Fneu` (`[fluorescence] neuropil_coefficient`; the QC notebook's standard) |
| Neuropil Fitted | `F − cᵢ·Fneu`, each ROI's own `cᵢ`: the least-squares slope of F on its neuropil outside its transients, kept in [0, 1] |
| No Neuropil | raw `F` |
| No Neuropil, Bg Regressed | raw `F` with the background (vessel mask, or every pixel outside the ROIs) regressed out per ROI, mean kept |
| Neuropil 0.7, Bg Regressed | background regressed out of `F` and `Fneu`, then `F − 0.7·Fneu` (the QC notebook's background version) |

ΔF/F0 then uses the same baseline for all (`[fluorescence] baseline`, `win_baseline_s`, ...). Before that, two diagnostics:

1. **ROI contrast:** median F over median neuropil per ROI. Near 1 means the cells barely stand out from their surroundings (expression, indicator brightness), whatever the correction.
2. **Green vs red background:** whether the field-wide fluctuation is in both channels by the same fraction (laser or shared detection: hardware), green more than red (blood volume), or green only (indicator signal, e.g. out-of-focus somata).

It uses the `[qc]` section of the config (the run, `cellpose_seg`, `background_seg`, `valid_frames`) and reads only. With `SAVE_FIGURES = True`, each figure is saved as png, pdf and svg with a `.caption.txt` in `<plane>/extraction_comparison/`.

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mini2p_preprocessing import plots
from mini2p_preprocessing.background_io import SOURCE_LABELS, load_non_roi_traces, load_vessel_traces
from mini2p_preprocessing.config import (
    distortion_corrected,
    find_registered_plane,
    load_qc_run,
    load_settings,
    variant_output_dir,
)
from mini2p_preprocessing.extraction_comparison import (
    background_channel_check,
    contrast_ratio,
    corrected_traces,
    fitted_neuropil_coefficients,
    method_metrics,
)
from mini2p_preprocessing.fluorescence import dff_from_corrected, select_frames
from mini2p_preprocessing.pipeline import QC_FONTS
from mini2p_preprocessing.plane import load_plane_traces, load_valid_frames
from mini2p_preprocessing.registration import load_registered_outputs

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s", force=True)

CONFIG = Path("../config/pipeline.toml")
SAVE_FIGURES = True  # png, pdf and svg with a caption, in <plane>/extraction_comparison/

settings = load_settings(CONFIG)
run = load_qc_run(CONFIG)
fl = settings.fluorescence
OUTPUT_DIR = variant_output_dir(
    run.run_dir, settings.destripe.enabled, distortion_corrected=distortion_corrected(settings)
)
PLANE_DIR = find_registered_plane(OUTPUT_DIR, settings.registration.motion_alignment_source,
                                  settings.registration.save_folder)
if PLANE_DIR is None:
    raise FileNotFoundError(f"No registration in {OUTPUT_DIR}; register it in the QC notebook")
FIGURES = PLANE_DIR / "extraction_comparison"

traces = load_plane_traces(PLANE_DIR)
frames = load_valid_frames(run.valid_frames, run.valid_frames_key) if run.valid_frames else None
f, fneu = select_frames(traces.f, frames), select_frames(traces.fneu, frames)
print("Plane:", PLANE_DIR)
print(f"{f.shape[0]} ROIs x {f.shape[1]} frames "
      f"({'valid frames only' if frames is not None else 'all frames'}) at {traces.fs:g} Hz")

registered = load_registered_outputs(PLANE_DIR)
backgrounds = {}  # label -> BackgroundTraces (cached in the plane folder after the first run)
if run.background_seg is None:
    print("No [qc] background_seg: the vessel background is skipped")
else:
    backgrounds["Vessel"] = load_vessel_traces(registered, PLANE_DIR, run.background_seg, settings,
                                               image=run.background_image, exclude=traces.labels > 0)
backgrounds["Non-ROI"] = load_non_roi_traces(registered, PLANE_DIR, traces.labels, settings)
for label, background in backgrounds.items():
    print(f"{label} background: {int(background.mask.sum())} pixels")


def save(fig, name, caption):
    """The figure as png, pdf and svg, and its caption, in FIGURES (if SAVE_FIGURES)."""
    if not SAVE_FIGURES:
        return
    FIGURES.mkdir(exist_ok=True)
    for extension in ("png", "pdf", "svg"):
        fig.savefig(FIGURES / f"{name}.{extension}", dpi=200, bbox_inches="tight")
    (FIGURES / f"{name}.caption.txt").write_text(caption.strip() + "\n")

## 1. How far the ROIs stand above their surroundings

Median raw F over median neuropil, per ROI. With soma-targeted GCaMP a healthy cell should be clearly brighter than its ring (ratios well above 1); ratios near 1 mean the cells barely stand out, which no correction can fix. The raw levels are printed too: if the neuropil's level is near zero or negative, the detector offset is part of the problem (a NaN ratio means a neuropil median ≤ 0).

In [ ]:
ratio = contrast_ratio(f, fneu)
print(f"Median F per ROI: {np.median(f):.1f} (5th-95th percentile over ROIs "
      f"{np.percentile(np.median(f, axis=1), 5):.1f}-{np.percentile(np.median(f, axis=1), 95):.1f})")
print(f"Median neuropil per ROI: {np.median(fneu):.1f}")
print(f"Lowest raw F anywhere: {f.min():.1f}; lowest neuropil: {fneu.min():.1f}")
print(f"ROIs with no positive neuropil median: {int(np.isnan(ratio).sum())}")
for cut in (1.1, 1.25, 1.5):
    print(f"ROIs with contrast below {cut}: {int((ratio < cut).sum())} of {len(ratio)}")
with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": QC_FONTS}):
    ax = plots.plot_contrast_ratio(ratio)
    save(ax.figure, "roi_contrast", """
Median raw F over median neuropil (Fneu) for each ROI, over the frames analysed. Dashed: 1,
a ROI no brighter than its surrounding neuropil ring. ROIs whose neuropil median is not
positive have no ratio and are left out (n counts the rest).""")
    plt.show()

## 2. The field-wide fluctuation: green and red

Each background trace as a fraction of its mean, green and red, and red's fractional change against green's. How to read the slope (red on green):

- **near 1**, with high r: both channels move by the same fraction, so the cause is upstream of the indicator and common to both, e.g. laser power or shared detection: a **hardware** problem;
- **between 0 and 1**: green moves more than red, as with **blood volume** (hemoglobin absorbs green more than red);
- **near 0**: the fluctuation is in the green indicator only, e.g. **activity of out-of-focus somata**.

A noisy red channel also pulls the slope towards 0, so read it with r; the red channel was unreliable in some sessions.

In [ ]:
channel_checks = {}
for label, background in backgrounds.items():
    green_bg, red_bg = select_frames(background.green, frames), select_frames(background.red, frames)
    check = background_channel_check(green_bg, red_bg)
    channel_checks[label] = check
    print(f"{label}: red-on-green slope {check['red_vs_green_slope']:.2f}, r = {check['r']:.2f}, "
          f"SD green {100 * check['green_sd_fraction']:.1f}% / red {100 * check['red_sd_fraction']:.1f}% of the mean")
    with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": QC_FONTS}):
        axes = plots.plot_background_channels(green_bg, red_bg, traces.fs, check)
        axes[0].figure.suptitle(f"{label} Background")
        save(axes[0].figure, f"background_green_red_{label.lower().replace('-', '_')}", f"""
{label} background (mean of its pixels per frame), green and red, each as a fraction of its
own mean. Left: over time. Right: red against green (one point per frame, subsampled) with
the least-squares slope through zero (solid) and slope 1 (dashed). Slope near 1: both channels
change by the same fraction (laser or shared detection); between 0 and 1: green changes more
(e.g. blood volume); near 0: green only (indicator signal). A noisy red channel also lowers
the slope; r = {check['r']:.2f}.""")
        plt.show()

## 3. The corrections

`FIT_TRANSIENT_SD`: frames more than this many robust SDs above the fit are left out of each ROI's neuropil fit (its transients). `[fluorescence] min_f0` (here 0) is where ΔF/F0 becomes undefined. Computing every baseline takes a minute or two.

In [ ]:
FIT_TRANSIENT_SD = 2.0

fitted = fitted_neuropil_coefficients(f, fneu, transient_sd=FIT_TRANSIENT_SD)
print(f"Fitted neuropil coefficients: median {np.median(fitted):.2f} "
      f"[{np.percentile(fitted, 5):.2f}, {np.percentile(fitted, 95):.2f}] (5th-95th percentile); "
      f"{int((fitted == 0).sum())} at 0, {int((fitted == 1).sum())} at 1")
with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": QC_FONTS}):
    ax = plots.plot_fitted_coefficients(fitted, fl.neuropil_coefficient)
    save(ax.figure, "fitted_neuropil_coefficients", f"""
Each ROI's fitted neuropil coefficient: the least-squares slope of its raw F on its neuropil,
refitted three times without frames more than {FIT_TRANSIENT_SD:g} robust SDs above the fit
(its transients), kept in [0, 1]. Dashed: the fixed coefficient,
{fl.neuropil_coefficient:g}.""")
    plt.show()

background_green = {label: select_frames(b.green, frames) for label, b in backgrounds.items()}
corrected = corrected_traces(f, fneu, background_green, fl.neuropil_coefficient, fitted)
results = {name: dff_from_corrected(trace, traces.fs, fl) for name, trace in corrected.items()}
reference = background_green.get("Vessel", background_green["Non-ROI"])
metrics = {name: method_metrics(result, reference) for name, result in results.items()}

keys = list(next(iter(metrics.values())))
SHORT = {"rois_f0_le_0_anywhere": "F0≤0 somewhere", "rois_f0_le_0_median": "F0≤0 mostly",
         "fraction_frames_f0_le_0": "frames no ΔF/F0", "median_f0": "median F0",
         "median_noise": "noise", "median_skew": "skew", "median_abs_r_background": "|r| bg",
         "mean_pairwise_r": "pairwise r"}
width = max(len(name) for name in metrics)
print(f"\n{'':{width}}  " + "  ".join(f"{SHORT[k]:>15}" for k in keys))
for name, row in metrics.items():
    print(f"{name:{width}}  " + "  ".join(f"{row[k]:>15.3g}" for k in keys))
print(f"\n|r| with background and pairwise r use the {'vessel' if 'Vessel' in background_green else 'non-ROI'} background")

**How to read it.** A correction that suits soma-targeted GCaMP should leave few ROIs with F0 ≤ 0, keep positive skew (transients), and leave little of the background in the traces (low |r| with the background, low pairwise r). Noise is in ΔF/F0 units, so it is not comparable on its own: without neuropil subtraction F0 includes the surrounding light, which shrinks both the noise and the transients in ΔF/F0; skew does not depend on that scale. Removing too much shared signal can also remove real population activity, so a low pairwise r is not the goal on its own. If every correction leaves many ROIs at F0 ≤ 0 or the ROI contrast above is near 1, the cells are too dim relative to their surroundings, and the fix is expression or brightness, not analysis.

In [ ]:
with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": QC_FONTS}):
    ax = plots.plot_f0_by_method({name: np.median(r.f0, axis=1) for name, r in results.items()})
    save(ax.figure, "f0_by_correction", """
Cumulative distribution over ROIs of each ROI's median baseline F0, for each correction (the
same ROIs and frames, the same baseline method and settings). Dashed: F0 = 0, at and below
which ΔF/F0 is undefined.""")
    plt.show()
    axes = plots.plot_method_metrics(metrics, keys)
    axes[0].figure.suptitle("Corrections Compared")
    save(axes[0].figure, "corrections_compared", """
For each correction, on the same ROIs: ROIs whose F0 is <= 0 in any frame and in at least
half of the frames; the fraction of ROI-frames without a ΔF/F0; the median over ROIs of each
ROI's median F0; the median frame-to-frame SD of ΔF/F0 (robust SD of frame differences /
sqrt 2); the median skewness of ΔF/F0; the median |r| between each ROI's corrected F and the
background (vessel if there is one, else non-ROI); the mean correlation between ROIs'
corrected F (up to 300 ROIs, evenly spaced).""")
    plt.show()

## 4. The same ROIs under each correction

The `N_SHOWN` ROIs with the largest 99th percentile of ΔF/F0 under `RANK_BY`, among those whose F0 stays above 0 there, drawn the same way under every correction on one vertical scale. A ROI whose F0 is ≤ 0 under a correction has gaps there.

In [ ]:
N_SHOWN = 12
RANK_BY = next((name for name in results if name.startswith("No Neuropil, ")), "Neuropil Fitted")
SHOWN_METHODS = list(results)  # or a few names from the table above

ranked = results[RANK_BY]
valid = ~ranked.invalid_f0.any(axis=1)
p99 = np.full(len(valid), -np.inf)
p99[valid] = np.nanpercentile(ranked.dff[valid], 99, axis=1)
rois = np.argsort(-p99, kind="stable")[: min(N_SHOWN, int(valid.sum()))]
print(f"Ranked by {RANK_BY}: ROIs {rois.tolist()}")
with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": QC_FONTS}):
    axes = plots.plot_top_rois({name: results[name].dff for name in SHOWN_METHODS}, rois,
                               xlabel="Valid Frames" if frames is not None else "Frames")
    axes[0].figure.suptitle(f"Top {len(rois)} ROIs By 99th Percentile ΔF/F0 ({RANK_BY})")
    save(axes[0].figure, "top_rois_by_correction", f"""
The {len(rois)} ROIs with the largest 99th percentile of ΔF/F0 under "{RANK_BY}", among
ROIs whose F0 stays above 0 under it, shown under each correction (one panel each) on one
vertical scale (bar: one trace spacing, in ΔF/F0). Gaps: frames where that correction's F0
is <= 0.""")
    plt.show()